In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col,
    lit,
    trim,
    upper,
    lower,
    coalesce,
    nullif,
    when,
    current_timestamp,
    concat_ws,
    sha2,
    object_construct,
    row_number
)
from snowflake.snowpark import Window
from datetime import datetime
import uuid

session = get_active_session()

exec(open('../DWH/parameters.py').read())

In [ ]:
%%sql -r dataframe_1
USE DATABASE IDENTIFIER('{{DB}}');
USE WAREHOUSE IDENTIFIER('{{WH}}');
USE SCHEMA IDENTIFIER('{{RAW}}');

In [ ]:
def has_data(stream):
    result = session.sql(
        f"SELECT SYSTEM$STREAM_HAS_DATA('{DB}.{RAW}.{stream}')"
    ).collect()[0][0]
    print(f"Checking stream: {DB}.{RAW}.{stream} = {result}")
    return str(result).lower() == 'true'

In [ ]:
# 1. ALLOWABLE_VALUES - no dependencies
if has_data(STREAM_T_ALLOWABLE_VALUES):
    %run ../Staging/STG_ALLOWABLE_VALUES.ipynb
else:
    print("no streams data")

In [ ]:
# 2. PERSONS - no dependencies
if has_data(STREAM_T_PERSONS):
    %run ../Staging/STG_PERSONS.ipynb
    %run ../DWH/DIM_T_PERSONS.ipynb
else:
    print("no streams data")

In [ ]:
# 3. MEDICATIONS - no dependencies
if has_data(STREAM_T_MEDICATIONS):
    %run ../Staging/STG_MEDICATIONS.ipynb
    %run ../DWH/DIM_MEDICATIONS_INFO.ipynb
    %run ../DWH/FACT_MEDICATIONS.ipynb
else:
    print("no streams data")

In [ ]:
# 4. CASES - depends on PERSONS
if has_data(STREAM_T_CASES):
    %run ../Staging/STG_CASES.ipynb
    %run ../DWH/DIM_CASES_INFO.ipynb
    %run ../DWH/FACT_CASES.ipynb
else:
    print("no streams data")

In [ ]:
# 5. PERSON_ORG_INVOLVEMENT - depends on PERSONS, CASES
if has_data(STREAM_T_PERSON_ORG_INVOLVEMENT):
    %run ../Staging/STG_PERSON_ORG_INVOLVEMENT.ipynb
    %run ../DWH/DIM_PERSON_ORG_INVOLVEMENT_INFO.ipynb
    %run ../DWH/FACT_PERSON_ORG_INVOLVEMENT.ipynb
    
else:
    print("no streams data")

In [ ]:
# 6. CUSTODIES - depends on PERSONS, PERSON_ORG_INVOLVEMENT
if has_data(STREAM_T_CUSTODIES):
    %run ../Staging/STG_CUSTODIES.ipynb
    %run ../DWH/DIM_CUSTODIES_INFO.ipynb
    %run ../DWH/FACT_CUSTODIES.ipynb
    
else:
    print("no streams data")

In [ ]:
# 7. PERSON_ROLE_ASSIGNMENTS - depends on PERSON_ORG_INVOLVEMENT
if has_data(STREAM_T_PERSON_ROLE_ASSIGNMENTS):
    %run ../Staging/STG_PERSON_ROLE_ASSIGNMENTS.ipynb
    %run ../DWH/DIM_PERSON_ROLE_ASSIGNMENTS_INFO.ipynb
    %run ../DWH/FACT_PERSON_ROLE_ASSIGNMENTS.ipynb
    
else:
    print("no streams data")

In [ ]:
# 8. MEDICATION_HEALTH_BEHAVIOR - depends on MEDICATIONS, PERSONS
if has_data(STREAM_T_MEDICATION_HEALTH_BEHAVIOR):
    %run ../Staging/STG_MEDICATION_HEALTH_BEHAVIOR.ipynb
    %run ../DWH/DIM_MEDICATION_HEALTH_BEHAVIOR_INFO.ipynb
    %run ../DWH/FACT_MEDICATION_HEALTH_BEHAVIOR.ipynb
else:
    print("no streams data")

In [ ]:
# 9. DATAMART 
%run ../Datamart/DIM_DM_CAS_CASES_DT.ipynb
%run ../Datamart/DIM_DM_CUS_CUSTODIES_DT.ipynb
%run ../Datamart/DIM_DM_MEDICATION_CONSENT_DETAILS_DT.ipynb
%run ../Datamart/FACT_DM_MEDICATION_CONSENT.ipynb
print("Datamart layer complete")